# BASE LIGHTGBM BENCHMARK V1

This notebook contains LightGBM model setup for generating competition submissions using **Polars** for optimized performance

**Best Model:** LightGBM Float32 **without clipping**
- **Score:** 0.218613 (better than with clipping: 0.217232)
- **Training time:** 53.82 seconds
- **Data:** Processed Float32 without clipping (v1)

**Purpose:**
Train LightGBM model and generate submission files for Kaggle competition

**Polars Benefits:**
- Faster data loading and processing
- Memory efficient operations
- Lazy evaluation capabilities
- Better multi-threading performance

## 1.1 IMPORTS AND CONFIGURATION

Setting up the environment with all necessary libraries and Polars configuration for optimal performance.

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time
from collections import Counter
import lightgbm as lgb

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "processed" / "train_processed_v1.parquet"  # Float32 without clipping
# Test data options - choose one:
# Option 1: Original test data (Float64)
full_test_path = base_dir / "data" / "test.parquet"
# Option 2: Processed test data (Float32) - uncomment to use
# full_test_path = base_dir / "data" / "processed" / "test_processed_v1.parquet"

results_dir = base_dir / "Results"

print("✅ Imports and configuration complete")
print(f"📁 Results directory: {results_dir}")
print(f"🚂 Train data: {full_train_path}")
print(f"🧪 Test data: {full_test_path}")

✅ Imports and configuration complete
📁 Results directory: ..\Results
🚂 Train data: ..\data\processed\train_processed_v1.parquet
🧪 Test data: ..\data\test.parquet


## 1.2 DATA LOADING

Loading training dataset (Float32 without clipping) and test dataset for submission.

In [2]:
# ============================================
# LOAD DATASETS
# ============================================
print("="*60)
print("LOADING DATASETS")
print("="*60)

# Load training dataset (Float32 without clipping)
start_time = time.time()
train_full = pl.read_parquet(full_train_path)
load_time = time.time() - start_time

print(f"Train full: {train_full.shape}")
print(f"Load time: {load_time:.2f} seconds")

# Load test dataset
test_full = pl.read_parquet(full_test_path)
print(f"Test full: {test_full.shape}")

print(f"\n✅ Datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING DATASETS
Train full: (5337414, 94)
Load time: 1.20 seconds
Test full: (1447107, 92)

✅ Datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## 2.0 DATA STRUCTURE ANALYSIS

Quick overview of dataset structure and data types.

In [3]:
# ============================================
# DATA STRUCTURE ANALYSIS
# ============================================
print("="*60)
print("DATA STRUCTURE ANALYSIS")
print("="*60)

# Train dataset structure
print("\n TRAIN DATASET STRUCTURE:")
print(f"Shape: {train_full.shape}")
print(f"\nData types:")
train_dtype_counts = Counter(str(dtype) for dtype in train_full.dtypes)
for dtype, count in train_dtype_counts.items():
    print(f"{dtype}: {count}")

# Test dataset structure  
print("\n TEST DATASET STRUCTURE:")
print(f"Shape: {test_full.shape}")
print(f"\nData types:")
test_dtype_counts = Counter(str(dtype) for dtype in test_full.dtypes)
for dtype, count in test_dtype_counts.items():
    print(f"{dtype}: {count}")

# Feature columns
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
print(f"\n🔢 Feature columns: {len(feature_cols)}")
print(f"📋 Sample features: {feature_cols[:5]}...")

DATA STRUCTURE ANALYSIS

 TRAIN DATASET STRUCTURE:
Shape: (5337414, 94)

Data types:
String: 1
Categorical: 3
Int16: 2
Float32: 86
Float64: 2

 TEST DATASET STRUCTURE:
Shape: (1447107, 92)

Data types:
String: 4
Int32: 3
Float64: 84
Int64: 1

🔢 Feature columns: 86
📋 Sample features: ['feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e']...


## 3.0 LIGHTGBM MODEL TRAINING

Training LightGBM model with Float32 data without clipping (best configuration).

In [9]:
# ============================================
# LIGHTGBM MODEL TRAINING
# ============================================

print("="*60)
print("TRAINING LIGHTGBM MODEL")
print("="*60)

# Metryka
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# Feature selection
feature_cols = [col for col in train_full.columns if col.startswith('feature_')]
X = train_full.select(feature_cols).fill_null(0).to_numpy()
y = train_full.select("y_target").to_numpy().ravel()
w = train_full.select("weight").to_numpy().ravel()

print(f"Features: {len(feature_cols)}")
print(f"X shape: {X.shape}")
print(f"X dtype: {X.dtype}")
print(f"Y shape: {y.shape}")
print(f"Weights shape: {w.shape}")
# LightGBM model - better parameters
start_time = time.time()
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',

    # Better parameters for quality
    num_leaves=50,           # Increase from 31
    learning_rate=0.05,        # Decrease from 0.1
    n_estimators=300,         # Increase from 100
    max_depth=10,             # Add depth control
    min_child_samples=20,       # Add regularization
    subsample=0.8,            # Add bagging
    colsample_bytree=0.8,       # Add feature sampling
    reg_alpha=0.1,            # Add L1 regularization
    reg_lambda=0.1,            # Add L2 regularization

    random_state=42,
    verbose=-1
)

lgb_model.fit(X, y, sample_weight=w)
lgb_time = time.time() - start_time


TRAINING LIGHTGBM MODEL
Features: 86
X shape: (5337414, 86)
X dtype: float32
Y shape: (5337414,)
Weights shape: (5337414,)


In [11]:
# Predykcja i score
y_pred_lgb = lgb_model.predict(X)
lgb_score = weighted_rmse_score(y, y_pred_lgb, w)

print(f"LightGBM training time: {lgb_time:.2f} seconds")
print(f"LightGBM Score: {lgb_score:.6f}")
print(f"\n🎯 Best model ready!")

LightGBM training time: 135.23 seconds
LightGBM Score: 0.266488

🎯 Best model ready!


LightGBM training time: 117.17 seconds
LightGBM Score: 0.266303

🎯 Best model ready!

## 4.0 GENERATE SUBMISSION

Generate predictions on test data and create submission file for Kaggle.

In [13]:
# ============================================
# GENERATE SUBMISSION
# ============================================

print("="*60)
print("GENERATING SUBMISSION")
print("="*60)

# Prepare test features
X_test = test_full.select(feature_cols).fill_null(0).to_numpy()
print(f"Test features shape: {X_test.shape}")
print(f"Test features dtype: {X_test.dtype}")

# Generate predictions
start_time = time.time()
test_predictions = lgb_model.predict(X_test)
pred_time = time.time() - start_time

print(f"Prediction time: {pred_time:.2f} seconds")
print(f"Predictions shape: {test_predictions.shape}")
print(f"Predictions range: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")

# Create submission dataframe
submission_df = test_full.select(['id']).with_columns(
    pl.Series(name="prediction", values=test_predictions)
)

print(f"\nSubmission shape: {submission_df.shape}")
print("Sample predictions:")
print(submission_df.head())

GENERATING SUBMISSION
Test features shape: (1447107, 86)
Test features dtype: float64
Prediction time: 3.43 seconds
Predictions shape: (1447107,)
Predictions range: [-100.137282, 69.679035]

Submission shape: (1447107, 2)
Sample predictions:
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ id                              ┆ prediction │
│ ---                             ┆ ---        │
│ str                             ┆ f64        │
╞═════════════════════════════════╪════════════╡
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.073733  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.170919  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.144553  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.011946  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.218893  │
└─────────────────────────────────┴────────────┘


## 5.0 SAVE SUBMISSION FILE

Save submission file in Results directory.

In [14]:
# ============================================
# SAVE SUBMISSION FILE
# ============================================

print("="*60)
print("SAVING SUBMISSION FILE")
print("="*60)

# Generate timestamp for filename
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"lightgbm_benchmark_v1_{timestamp}.csv"
submission_path = results_dir / submission_filename

# Save submission
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 File size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"📝 Records: {len(submission_df)}")

# Verify file exists and show sample
if submission_path.exists():
    print("\n📋 File verification:")
    verify_df = pl.read_csv(submission_path)
    print(f"Loaded shape: {verify_df.shape}")
    print("Sample:")
    print(verify_df.head())
else:
    print("❌ File not found!")

SAVING SUBMISSION FILE
✅ Submission saved: ..\Results\lightgbm_benchmark_v1_20260318_032031.csv
📊 File size: 82.27 MB
📝 Records: 1447107

📋 File verification:
Loaded shape: (1447107, 2)
Sample:
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ id                              ┆ prediction │
│ ---                             ┆ ---        │
│ str                             ┆ f64        │
╞═════════════════════════════════╪════════════╡
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.073733  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.170919  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.144553  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.011946  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.218893  │
└─────────────────────────────────┴────────────┘


## 6.0 SUMMARY

Model training and submission generation complete!

In [15]:
# ============================================
# SUMMARY
# ============================================

print("="*60)
print("LIGHTGBM BENCHMARK V1 SUMMARY")
print("="*60)

print(f"🎯 Model: LightGBM Float32 (without clipping)")
print(f"📊 Training Score: {lgb_score:.6f}")
print(f"⏱️ Training Time: {lgb_time:.2f} seconds")
print(f"📁 Data: Train v1 (Float32, no clipping)")
print(f"🔢 Features: {len(feature_cols)}")
print(f"📏 Train size: {train_full.shape}")
print(f"📏 Test size: {test_full.shape}")
print(f"💾 Submission: {submission_path}")
print(f"📊 Submission size: {submission_path.stat().st_size / 1024**2:.2f} MB")

print(f"\n🚀 Ready for Kaggle submission!")
print(f"\n📈 Model Performance Comparison:")
print(f"   - LightGBM (no clipping): {lgb_score:.6f} ⭐ BEST")
print(f"   - LightGBM (with clipping): 0.217232")
print(f"   - XGBoost (with clipping): 0.179537")
print(f"   - Linear Regression (with clipping): 0.046329")

print(f"\n✨ LightGBM without clipping is the winner!")
print(f"\n💡 Data Type Note:")
print(f"   - Train: Float32 (processed v1)")
print(f"   - Test: Float64 (original) - automatically converted during prediction")
print(f"   - You can switch to test_processed_v1.parquet for Float32 consistency")

LIGHTGBM BENCHMARK V1 SUMMARY
🎯 Model: LightGBM Float32 (without clipping)
📊 Training Score: 0.266488
⏱️ Training Time: 135.23 seconds
📁 Data: Train v1 (Float32, no clipping)
🔢 Features: 86
📏 Train size: (5337414, 94)
📏 Test size: (1447107, 92)
💾 Submission: ..\Results\lightgbm_benchmark_v1_20260318_032031.csv
📊 Submission size: 82.27 MB

🚀 Ready for Kaggle submission!

📈 Model Performance Comparison:
   - LightGBM (no clipping): 0.266488 ⭐ BEST
   - LightGBM (with clipping): 0.217232
   - XGBoost (with clipping): 0.179537
   - Linear Regression (with clipping): 0.046329

✨ LightGBM without clipping is the winner!

💡 Data Type Note:
   - Train: Float32 (processed v1)
   - Test: Float64 (original) - automatically converted during prediction
   - You can switch to test_processed_v1.parquet for Float32 consistency


In [16]:
# Sprawdź range y_target w train
print(f"Y target range: [{y.min():.6f}, {y.max():.6f}]")
print(f"Y target mean: {y.mean():.6f}")
print(f"Y target std: {y.std():.6f}")

# Sprawdź range features w test
print(f"Test features range: [{X_test.min():.6f}, {X_test.max():.6f}]")

Y target range: [-2201.881578, 2314.411152]
Y target mean: -0.665905
Y target std: 32.527639
Test features range: [-3088.540886, 19536754.573118]
